# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malakanwarr/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. Finding: Random Forest Feature Importance for Health Score (Page 27).
 Where the label comes from: The 'Health Score' target is a manual composite metric created by the authors, built from Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts).  
 Does the validation design carry the claim?: No. The model claims Average Position and Impressions are the top predictors. However, because those exact metrics were used to calculate the target label, the model is simply reverse-engineering the human formula. This is target leakage (circular logic) rather than a newly discovered causal relationship.  

2. Finding: Growing content is longer, so expand thin pages (Page 6).  Where the label comes from: The 'growing' vs 'declining' label is calculated from a 30-day impression trend (growing means >10% increase).  
Does the validation design carry the claim?: No. The data shows growing content averages 3.2K words while declining content averages 2.3K words. However, this is an observational correlation. The study did not run a controlled test (adding words to a failing page to see if it recovers). The validation design supports prediction, but not the causal recommendation to "expand thin pages" to force growth.

In [1]:
import pandas as pd

# 1. Audit Checklist Output
print("--- ML-09 Audit Diagnostic ---")
print("Audit 1 (Health Score): FAILED.")
print("Reason: Target Label (Health Score) is a direct mathematical function of the input features (Impressions, Position). This causes 100% Target Leakage.\n")

print("Audit 2 (Word Count vs. Growth): FAILED.")
print("Reason: Observational data (cross-sectional snapshot) is being used to make a causal intervention claim (expanding word counts). No A/B test or temporal intervention was measured.")

--- ML-09 Audit Diagnostic ---
Audit 1 (Health Score): FAILED.
Reason: Target Label (Health Score) is a direct mathematical function of the input features (Impressions, Position). This causes 100% Target Leakage.

Audit 2 (Word Count vs. Growth): FAILED.
Reason: Observational data (cross-sectional snapshot) is being used to make a causal intervention claim (expanding word counts). No A/B test or temporal intervention was measured.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My model under an honest split (before/after):
In Week 5, I built a K-Means clustering model to identify "Missed Opportunity" archetypes (high volume, low clicks) without using hard thresholds.

The Naive Way (Random Split): If I split the data randomly, pages from the same client leak across both the training and test sets. The model can artificially score higher by memorizing the structural quirks of specific websites.

The Honest Way (Grouped by Client): By using GroupShuffleSplit on client_hash_id, the model is forced to define its archetypes on one set of clients and prove they exist on completely unseen websites. The grouped score is the honest measurement of how well these archetypes generalize across the internet.

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# 1. Load data and setup the target definition (Week 4 rule)
df = pd.read_csv('master_dataset_ready.csv', low_memory=False)
df['target'] = ((df['search_volume'] >= 1000) & (df['gsc_clicks'] <= 5)).astype(int)
feature_cols = ['search_volume', 'gsc_clicks', 'competition', 'keyword_char_count']

# Helper function to train K-Means and return Precision@50
def get_kmeans_precision(train_df, test_df):
    scaler = StandardScaler()
    kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')

    # Train scaler and model
    X_train = scaler.fit_transform(train_df[feature_cols].fillna(0))
    kmeans.fit(X_train)
    train_df['cluster'] = kmeans.labels_

    # Find the "Missed Opportunity" archetype cluster (highest target rate)
    target_cluster = train_df.groupby('cluster')['target'].mean().idxmax()

    # Test on unseen data (calculate distance to the archetype center)
    X_test = scaler.transform(test_df[feature_cols].fillna(0))
    test_distances = kmeans.transform(X_test)[:, target_cluster]
    test_df['proximity_score'] = -test_distances # Closer is better

    # Calculate Precision@50
    top_50 = test_df.sort_values(by='proximity_score', ascending=False).head(50)
    return top_50['target'].mean()

# 2. The Naive Split (Random Shuffling = Leakage)
train_naive, test_naive = train_test_split(df, test_size=0.2, random_state=42)
naive_score = get_kmeans_precision(train_naive.copy(), test_naive.copy())

# 3. The Honest Split (Grouped by Client = No Leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
grouped_score = get_kmeans_precision(df.iloc[train_idx].copy(), df.iloc[test_idx].copy())

print("--- Model Validation: Split Comparison ---")
print(f"Naive Random Split (Precision@50): {naive_score:.2%}")
print(f"Honest Grouped Split (Precision@50): {grouped_score:.2%}")

--- Model Validation: Split Comparison ---
Naive Random Split (Precision@50): 100.00%
Honest Grouped Split (Precision@50): 46.00%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage Audit:
For my K-Means clustering model, I used a strict subset of four features: search_volume, gsc_clicks, competition, and keyword_char_count.

Product Flag Leakage: I confirmed that no internal backend flags (like is_deleted or is_published) were included in the clustering feature set.

Future Window Leakage: The dataset is bounded strictly to the March snapshot. There are no features representing future clicks, impressions, or May/June performance metrics. The model is forced to cluster based strictly on the current state of the page.

In [3]:
# 1. Define the exact features fed into the model
final_features = ['search_volume', 'gsc_clicks', 'competition', 'keyword_char_count']

print("--- Leakage Audit: Feature Set ---")
print(f"Features in use: {final_features}\n")

# 2. Scan for product flags (cheating by looking at backend database status)
suspicious_flags = [col for col in final_features if col.startswith('is_')]
if len(suspicious_flags) > 0:
    print(f"FAILED: Found suspicious product flags: {suspicious_flags}")
else:
    print("PASS: No product flags detected in feature set.")

# 3. Scan for time/future leakage (cheating by looking ahead in time)
time_flags = [col for col in final_features if 'date' in col or 'future' in col or 'next' in col]
if len(time_flags) > 0:
    print(f"FAILED: Found potential future-window leakage: {time_flags}")
else:
    print("PASS: No time or future-window data detected in feature set.")

--- Leakage Audit: Feature Set ---
Features in use: ['search_volume', 'gsc_clicks', 'competition', 'keyword_char_count']

PASS: No product flags detected in feature set.
PASS: No time or future-window data detected in feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The Dangerous (Bold) Claim: "My K-Means clustering model proves that high search volume combined with low clicks always creates a 'Missed Opportunity' archetype, and updating these pages will guarantee an increase in organic traffic."

The Safe (Audited) Claim: "We observed a distinct 'Missed Opportunity' archetype characterized by high search volume and low clicks. Using an honest grouped-client split, we measured the model's Precision@50 at 46%. While this provides a strong directional hint that these pages are underperforming relative to their potential, this model is a decision-support tool meant to help the content team prioritize their review queue, rather than a guarantee of future traffic."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.